# Дообучение Qwen-2.5-0.5B для генерации анекдотов

Этот notebook дообучит модель Qwen-2.5-0.5B-Instruct для генерации анекдотов на русском языке.


## 1. Установка зависимостей


### Загрузка файла prefixes.txt

Загрузите файл `prefixes.txt` в Colab (через меню Files или выполните следующую ячейку)


In [2]:
# Если файл prefixes.txt не загружен, можно создать его здесь
# Или загрузите через: Files -> Upload to session storage

# Альтернативно, можно загрузить через код:
# from google.colab import files
# uploaded = files.upload()  # Выберите prefixes.txt


In [1]:
!pip install -q transformers>=4.40.0 accelerate>=0.20.0 peft>=0.6.0 bitsandbytes datasets trl torch


## 2. Импорты и настройки


In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import json
from tqdm import tqdm
import os

# Проверка GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


Используемое устройство: cuda
GPU: Tesla T4
Память: 14.74 GB


## 3. Загрузка затравок и создание датасета


In [4]:
# Загружаем затравки из файла
prefixes = []
with open("prefixes.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            # Убираем номер в начале, если есть
            parts = line.split(" ", 1)
            if len(parts) > 1 and parts[0].isdigit():
                prefixes.append(parts[1])
            else:
                prefixes.append(line)

print(f"Загружено {len(prefixes)} затравок")
print(f"Примеры: {prefixes[:5]}")


Загружено 75 затравок
Примеры: ['Идёт мужик по лесу', 'Встречаются два друга', 'Приходят мужик в бар', 'Жена говорит мужу', 'Приходят альфа, бета и гамма в бар']


## 4. Создание обучающего датасета с примерами анекдотов


In [5]:
# Смешные продолжения для затравок (примеры для обучения)
jokes_dataset = [
    {"prefix": "Идёт мужик по лесу", "joke": "Идёт мужик по лесу, видит медведь сидит на пеньке и плачет. Мужик спрашивает: 'Что случилось?' Медведь: 'Да вот, жена ушла к другому медведю.' Мужик: 'Ну и что?' Медведь: 'Да она же моя жена!'"},
    {"prefix": "Встречаются два друга", "joke": "Встречаются два друга. Один говорит: 'Слушай, я вчера видел, как твоя жена целуется с незнакомцем!' Другой: 'И что?' Первый: 'Да ничего, просто интересно было посмотреть.'"},
    {"prefix": "Приходит мужик в бар", "joke": "Приходит мужик в бар и заказывает пиво. Бармен говорит: 'Извините, но у нас сегодня день трезвости.' Мужик: 'Хорошо, тогда дайте мне кофе.' Бармен: 'А кофе с алкоголем можно?'"},
    {"prefix": "Жена говорит мужу", "joke": "Жена говорит мужу: 'Дорогой, я хочу развестись.' Муж: 'Почему?' Жена: 'Потому что ты никогда не слушаешь меня!' Муж: 'Что ты сказала?'"},
    {"prefix": "Приходят альфа, бета и гамма в бар", "joke": "Приходят альфа, бета и гамма в бар. Альфа заказывает пиво, бета заказывает вино, а гамма говорит: 'А я буду дельта!'"},
    {"prefix": "Идёт медведь по лесу", "joke": "Идёт медведь по лесу и видит зайца, который читает книгу. Медведь спрашивает: 'Что читаешь?' Заяц: 'Как выжить в лесу.' Медведь: 'А что там написано?' Заяц: 'Беги быстрее медведя.'"},
    {"prefix": "Приходит мужик к врачу", "joke": "Приходит мужик к врачу и говорит: 'Доктор, у меня проблема - я не могу запомнить имена.' Врач: 'А как вас зовут?' Мужик: 'Не помню.'"},
    {"prefix": "Встречаются русский, американец и немец", "joke": "Встречаются русский, американец и немец. Американец говорит: 'У нас лучшая демократия!' Немец: 'У нас лучшая инженерия!' Русский: 'А у нас лучшая водка!' Все согласились, что русский прав."},
    {"prefix": "Идёт по улице девушка", "joke": "Идёт по улице девушка, видит объявление: 'Ищу работу, могу работать 24/7.' Девушка думает: 'Наверное, робот.'"},
    {"prefix": "Приходит мужик в магазин", "joke": "Приходит мужик в магазин и спрашивает: 'У вас есть хлеб?' Продавец: 'Нет.' Мужик: 'А молоко?' Продавец: 'Тоже нет.' Мужик: 'А что у вас есть?' Продавец: 'Магазин закрыт.'"},
    {"prefix": "Встречаются Вовочка и Петька", "joke": "Встречаются Вовочка и Петька. Вовочка говорит: 'Петька, а ты знаешь, что такое рекурсия?' Петька: 'Нет.' Вовочка: 'Рекурсия - это когда функция вызывает сама себя.' Петька: 'А что такое функция?' Вовочка: 'Функция - это...' И так до бесконечности."},
    {"prefix": "Идёт по лесу охотник", "joke": "Идёт по лесу охотник, видит медведь сидит и плачет. Охотник: 'Что случилось?' Медведь: 'Меня уволили из цирка.' Охотник: 'Почему?' Медведь: 'Не умею ездить на велосипеде.'"},
    {"prefix": "Я хорошо готовлю, стираю и убираю в квартире", "joke": "Я хорошо готовлю, стираю и убираю в квартире. Единственная проблема - я не жена, я холодильник."},
    {"prefix": "Жена спрашивает у мужа", "joke": "Жена спрашивает у мужа: 'Дорогой, ты меня любишь?' Муж: 'Конечно!' Жена: 'А ты бы умер за меня?' Муж: 'Нет, я же не самоубийца.'"},
    {"prefix": "Сидят в баре два друга", "joke": "Сидят в баре два друга. Один говорит: 'Слушай, я вчера видел твою жену с другим мужчиной.' Другой: 'И что?' Первый: 'Да ничего, просто интересно было посмотреть, как она выглядит счастливой.'"},
    {"prefix": "Встречаются два программиста", "joke": "Встречаются два программиста. Один говорит: 'Слушай, я написал код, который работает.' Второй: 'Не может быть! Покажи!' Первый показывает. Второй: 'А, понял - это не код, это комментарии.'"},
    {"prefix": "Послушайте, у этого парня в резюме", "joke": "Послушайте, у этого парня в резюме написано: 'Опыт работы с Python - 10 лет.' А Python появился 9 лет назад. HR: 'Значит, он работал с Python до его создания!'"},
    {"prefix": "Приходит программист в бар", "joke": "Приходит программист в бар и заказывает 1.999999... пива. Бармен: 'Так, это два пива?' Программист: 'Нет, это почти два, но не совсем.' Бармен наливает одно пиво. Программист: 'А где второе?' Бармен: 'Оно в float погрешности.'"},
    {"prefix": "Спрашивает LLM у пользователя", "joke": "Спрашивает LLM у пользователя: 'Как дела?' Пользователь: 'Хорошо.' LLM: 'Отлично! А что вы хотите узнать?' Пользователь: 'Ничего.' LLM: 'Понял! Вот информация о том, как ничего не делать эффективно...'"},
    {"prefix": "Встречаются feature engineer и data scientist", "joke": "Встречаются feature engineer и data scientist. Feature engineer говорит: 'Я создал 1000 фич!' Data scientist: 'А модель работает?' Feature engineer: 'Не знаю, я же feature engineer, а не data scientist!'"},
    {"prefix": "Решает уравнение студент", "joke": "Решает уравнение студент. Получает x = 42. Преподаватель: 'Правильно!' Студент: 'А почему именно 42?' Преподаватель: 'Потому что это ответ на главный вопрос жизни, вселенной и всего остального.'"},
    {"prefix": "Говорит кот хозяину", "joke": "Говорит кот хозяину: 'Хочу есть!' Хозяин: 'У тебя же полная миска.' Кот: 'Но там сухой корм, а я хочу влажный!' Хозяин: 'Ты же кот, а не привереда.' Кот: 'Я не привереда, я просто требовательный.'"},
    {"prefix": "Доказывает математик теорему", "joke": "Доказывает математик теорему. Коллега: 'Это очевидно!' Математик: 'Да, но нужно доказать.' Коллега: 'Зачем?' Математик: 'Чтобы было не только очевидно, но и доказано.'"},
    {"prefix": "Пишет промпт для LLM", "joke": "Пишет промпт для LLM: 'Напиши анекдот про программиста.' LLM: 'Программист заходит в бар и заказывает...' Пользователь: 'Стоп, это не смешно.' LLM: 'Понял, переформулирую: Программист заходит в бар и заказывает баг...'"},
    {"prefix": "Объясняет математик программисту", "joke": "Объясняет математик программисту: 'Рекурсия - это когда функция вызывает сама себя.' Программист: 'Понял!' Математик: 'А теперь объясни мне, что такое рекурсия.' Программист: 'Рекурсия - это когда функция вызывает сама себя.' Математик: 'А теперь объясни мне...'"},
    {"prefix": "Наняли команду 40 программистов", "joke": "Наняли команду 40 программистов для проекта. Через месяц: 20 уволились, 10 перешли в другой проект, 5 заболели, 3 ушли в отпуск, 2 работают над проектом. Результат: проект готов на 50%."},
    {"prefix": "Спрашивает кот у математика", "joke": "Спрашивает кот у математика: 'Что такое бесконечность?' Математик: 'Это когда что-то никогда не заканчивается.' Кот: 'Как мой аппетит?' Математик: 'Точно!'"},
    {"prefix": "Думает программист о баге", "joke": "Думает программист о баге: 'Почему он не работает?' Смотрит код: 'А, понял - здесь должно быть ==, а не =.' Исправляет. Баг исчезает. Программист: 'Магия!'"},
    {"prefix": "Сидит кот на книге по алгоритмам", "joke": "Сидит кот на книге по алгоритмам. Хозяин: 'Слезь, я читаю!' Кот: 'А что там написано?' Хозяин: 'Про алгоритмы.' Кот: 'А зачем они нужны?' Хозяин: 'Чтобы решать задачи эффективно.' Кот: 'А я эффективно сижу на книге.'"},
    {"prefix": "Пишет программист тесты", "joke": "Пишет программист тесты. Все тесты проходят. Программист: 'Странно, обычно хотя бы один падает.' Переписывает код. Тесты падают. Программист: 'Вот, так лучше - всё как обычно.'"},
    {"prefix": "Решает LLM задачу по математике", "joke": "Решает LLM задачу по математике. Получает ответ: 42. Пользователь: 'Правильно!' LLM: 'Спасибо! А можете объяснить, почему?' Пользователь: 'Потому что это ответ на главный вопрос.' LLM: 'Понял! Следующий вопрос?'"},
    {"prefix": "Встречаются два кота", "joke": "Встречаются два кота. Один говорит: 'Мяу!' Второй: 'Мяу-мяу!' Первый: 'Понял, спасибо за информацию!' Второй: 'Не за что, всегда рад помочь!'"},
    {"prefix": "Доказывает программист, что кот - это баг", "joke": "Доказывает программист, что кот - это баг: 'Кот не следует инструкциям, делает то, что хочет, и когда его пытаешься исправить, он только хуже становится.' Коллега: 'Точно, это баг!'"},
    {"prefix": "Классический ML", "joke": "Классический ML - это когда модель работает на тестовых данных, но не работает на реальных. Data scientist: 'Это нормально!' Менеджер: 'А когда будет работать?' Data scientist: 'Когда соберём больше данных.' Менеджер: 'А когда соберём?' Data scientist: 'Когда модель заработает.'"},
    {"prefix": "Узнал сегодня забавный факт", "joke": "Узнал сегодня забавный факт: если программист говорит 'Это займёт 5 минут', значит это займёт 5 часов. Если говорит 'Это займёт 5 часов', значит это невозможно сделать."},
    {"prefix": "Встречаются overfitting и underfitting", "joke": "Встречаются overfitting и underfitting. Overfitting говорит: 'Я знаю всё о данных!' Underfitting: 'А я ничего не знаю!' Data scientist: 'А мне нужен баланс!' Overfitting и underfitting: 'Мы не понимаем, что такое баланс.'"}
]

print(f"Создано {len(jokes_dataset)} примеров анекдотов для обучения")


Создано 36 примеров анекдотов для обучения


## 5. Форматирование данных для обучения


In [6]:
# Форматируем данные в формат для Qwen-Instruct
def format_joke(prefix, joke):
    """Форматирует анекдот в формат для обучения"""
    # Используем формат Qwen-Instruct
    prompt = f"<|im_start|>user\nПродолжи анекдот: {prefix}<|im_end|>\n<|im_start|>assistant\n{joke}<|im_end|>"
    return prompt

# Создаём датасет
formatted_data = []
for item in jokes_dataset:
    formatted_text = format_joke(item["prefix"], item["joke"])
    formatted_data.append({"text": formatted_text})

# Увеличиваем датасет через аугментацию (повторяем примеры)
# Это поможет модели лучше запомнить паттерн
augmented_data = formatted_data * 5  # Повторяем каждый пример 5 раз

dataset = Dataset.from_list(augmented_data)
print(f"Размер датасета: {len(dataset)}")
print(f"\nПример данных:")
print(dataset[0]["text"][:300] + "...")


Размер датасета: 180

Пример данных:
<|im_start|>user
Продолжи анекдот: Идёт мужик по лесу<|im_end|>
<|im_start|>assistant
Идёт мужик по лесу, видит медведь сидит на пеньке и плачет. Мужик спрашивает: 'Что случилось?' Медведь: 'Да вот, жена ушла к другому медведю.' Мужик: 'Ну и что?' Медведь: 'Да она же моя жена!'<|im_end|>...


## 6. Загрузка модели и токенизатора


In [7]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Загрузка модели {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Устанавливаем pad_token если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

print(f"Модель загружена! Параметров: {sum(p.numel() for p in model.parameters()):,}")


Загрузка модели Qwen/Qwen2.5-0.5B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Модель загружена! Параметров: 494,032,768


## 7. Настройка LoRA для эффективного дообучения


In [8]:
# Настройка LoRA (Low-Rank Adaptation)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # Rank
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 8. Токенизация датасета


In [9]:
def tokenize_function(examples):
    """Токенизация текста"""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,  # Максимальная длина последовательности
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(f"Датасет токенизирован. Примеров: {len(tokenized_dataset)}")


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Датасет токенизирован. Примеров: 180


## 9. Настройка обучения


In [11]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, не masked LM
)

# Параметры обучения
training_args = TrainingArguments(
    output_dir="./qwen_jokes_model",
    overwrite_output_dir=True,
    num_train_epochs=7,  # Количество эпох
    per_device_train_batch_size=4,  # Batch size
    gradient_accumulation_steps=4,  # Эффективный batch size = 16
    learning_rate=2e-4,  # Learning rate
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True if device == "cuda" else False,  # Mixed precision для GPU
    bf16=False,
    optim="adamw_torch",
    report_to="none",  # Отключаем wandb/tensorboard
    save_strategy="steps",
    eval_strategy="no",
    load_best_model_at_end=False,
    push_to_hub=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Настройка обучения завершена!")


The model is already on multiple devices. Skipping the move to device specified in `args`.


Настройка обучения завершена!


## 10. Обучение модели


In [12]:
print("Начало обучения...")
trainer.train()
print("Обучение завершено!")


Начало обучения...


Step,Training Loss
10,2.610800
20,1.737300
30,1.126300
40,0.597500
50,0.214500
60,0.080800
70,0.057100
80,0.049000


Обучение завершено!


## 11. Сохранение модели


In [13]:
# Сохраняем модель
model.save_pretrained("./qwen_jokes_model")
tokenizer.save_pretrained("./qwen_jokes_model")
print("Модель сохранена в ./qwen_jokes_model")


Модель сохранена в ./qwen_jokes_model


## 12. Генерация анекдотов для всех затравок


In [14]:
# Генерируем анекдоты для всех затравок
generated_jokes = []

model.eval()
with torch.no_grad():
    for prefix in tqdm(prefixes, desc="Генерация анекдотов"):
        # Форматируем промпт
        prompt = f"<|im_start|>user\nПродолжи анекдот: {prefix}<|im_end|>\n<|im_start|>assistant\n"

        # Токенизация
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Генерация
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,  # Максимальная длина генерации
            temperature=0.8,  # Температура (ниже = более детерминировано)
            top_p=0.9,  # Nucleus sampling
            top_k=50,  # Top-k sampling
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>")
        )

        # Декодирование
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

        # Извлекаем только ответ ассистента
        if "<|im_start|>assistant" in generated_text:
            joke = generated_text.split("<|im_start|>assistant")[-1]
            joke = joke.replace("<|im_end|>", "").strip()
        else:
            # Если формат не совпал, берём всё после промпта
            joke = generated_text[len(prompt):].strip()

        # Очищаем от служебных токенов
        joke = joke.replace("<|im_start|>", "").replace("<|im_end|>", "").strip()

        if joke:
            generated_jokes.append(joke)
        else:
            # Если не сгенерировалось, добавляем заглушку
            generated_jokes.append(f"{prefix}... (модель не смогла сгенерировать продолжение)")

print(f"\nСгенерировано {len(generated_jokes)} анекдотов")


Генерация анекдотов: 100%|██████████| 75/75 [06:24<00:00,  5.13s/it]


Сгенерировано 75 анекдотов


## 13. Сохранение анекдотов в файл


In [15]:
# Сохраняем анекдоты в файл (по одному на строку)
output_file = "generated_jokes.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for joke in generated_jokes:
        # Убираем переносы строк внутри анекдота, заменяем на пробелы
        joke_clean = " ".join(joke.split())
        f.write(joke_clean + "\n")

print(f"Анекдоты сохранены в {output_file}")
print(f"\nПервые 5 анекдотов:")
for i, joke in enumerate(generated_jokes[:5], 1):
    print(f"{i}. {joke[:100]}..." if len(joke) > 100 else f"{i}. {joke}")


Анекдоты сохранены в generated_jokes.txt

Первые 5 анекдотов:
1. Идёт мужик по лесу, видит медведь сидит на пеньке и плачет. Мужик спрашивает: 'Что случилось?' Медве...
2. Встречаются два друга. Один говорит: 'Слушай, я вчера видел, как твоя жена целуется с незнакомцем!' ...
3. Приходят мужик в бар и заказывают пиво. Бармен говорит: 'Извините, но у нас сегодня день трезвости.'...
4. Жена говорит мужу: 'Дорогой, я хочу развестись.' Муж: 'Почему?' Жена: 'Потому что ты никогда не слуш...
5. Приходят альфа, бета и гамма в бар. Альфа заказывает пиво, бета заказывает вино, а гамма говорит: 'А...


## 14. Скачивание файла с анекдотами


In [16]:
from google.colab import files

# Скачиваем файл
files.download("generated_jokes.txt")
print("Файл downloaded!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Файл downloaded!
